In [ ]:
import json
import os

merged_path = os.path.join(os.getcwd(), "merged_annotations.json")
with open(merged_path, "r", encoding="utf-8") as fp:
    coco = json.load(fp)

img_id_to_info = {img["id"]: img for img in coco["images"]}

invalid_size = []
out_of_bounds_minor = []
out_of_bounds_major = []

for ann in coco["annotations"]:
    x, y, w, h = ann["bbox"]
    img_info = img_id_to_info[ann["image_id"]]
    img_w, img_h = img_info["width"], img_info["height"]

    if w <= 0 or h <= 0:
        invalid_size.append(ann)
        continue

    x2, y2 = x + w, y + h
    over_left = max(0 - x, 0)
    over_top = max(0 - y, 0)
    over_right = max(x2 - img_w, 0)
    over_bottom = max(y2 - img_h, 0)
    max_overflow = max(over_left, over_top, over_right, over_bottom)

    if max_overflow > 0:
        threshold = 0.05 * max(img_w, img_h)
        if max_overflow <= threshold:
            out_of_bounds_minor.append((ann, max_overflow))
        else:
            out_of_bounds_major.append((ann, max_overflow))

print(f"전체 annotation 개수: {len(coco['annotations'])}")
print(f"width/height <= 0 인 경우: {len(invalid_size)}")
print(f"경계 살짝 벗어남 (clipping 후보): {len(out_of_bounds_minor)}")
print(f"경계 심하게 벗어남 (제거 후보): {len(out_of_bounds_major)}")

if out_of_bounds_major:
    print("\n=== 심하게 벗어난 bbox 샘플 ===")
    for ann, overflow in out_of_bounds_major[:5]:
        print(
            f"bbox={ann['bbox']}, overflow={overflow:.1f}, image_id={ann['image_id']}"
        )

# 이상치 제거 후 정제된 파일로 저장
clean_annotations = []
removed = []

for ann in coco["annotations"]:
    x, y, w, h = ann["bbox"]
    img_info = img_id_to_info[ann["image_id"]]
    img_w, img_h = img_info["width"], img_info["height"]
    x2, y2 = x + w, y + h
    over_right = max(x2 - img_w, 0)
    over_bottom = max(y2 - img_h, 0)
    over_left = max(0 - x, 0)
    over_top = max(0 - y, 0)
    max_overflow = max(over_left, over_top, over_right, over_bottom)
    threshold = 0.05 * max(img_w, img_h)

    if max_overflow > threshold:
        removed.append(ann)
        continue
    clean_annotations.append(ann)

coco["annotations"] = clean_annotations

output_path = os.path.join(os.getcwd(), "merged_annotations_clean.json")
with open(output_path, "w", encoding="utf-8") as fp:
    json.dump(coco, fp, ensure_ascii=False, indent=2)

print(f"\n제거된 annotation: {len(removed)}개")
print(f"정제된 annotation 개수: {len(clean_annotations)}")
print(f"저장 완료: {output_path}")
